# Agent with Memory Summarization
It uses summarization middleware to prevent memory from growing big

### Setup

Install Langchain dependencies

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-ollama

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_ollama import ChatOllama

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Instantiate the model

In [2]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Give the agent "memory"

In [3]:
checkpointer = InMemorySaver()

Create the agent

In [5]:
SYSTEM_PROMPT = "You are a helpful assistant"

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 800),
            keep=("messages", 4)
        )
    ],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

---

### Test

Let's try it out. Try to exceed the amount of tokens to trigger the summarization

In [6]:
config: RunnableConfig = {"configurable": {"thread_id": "001"}}

agent.invoke({"messages": "hi, my name is Tom"}, config)
agent.invoke({"messages": "write a 500 words story about about a boy named Jim's first day at a school named 'Cook State School'"}, config)
agent.invoke({"messages": "translate the story to Spanish"}, config)
agent.invoke({"messages": "translate the story to Italian"}, config)
final_response = agent.invoke({"messages": "what's my name? What was the boy's name? and the name of the school?"}, config)

In [7]:
import pprint
pprint.pprint(final_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user\'s primary goal is to translate the previously written story about a boy named Jim\'s first day at Cook State School from English into Spanish.\n\n## SUMMARY\nThe user initially tasked the AI with writing a creative story. The core parameters were: a 500-word story about a boy named Jim\'s first day at Cook State School. This story was successfully written and provided by the AI. The current and immediate goal is to translate this completed narrative into Spanish.\n\n## ARTIFACTS\nOne story artifact was created: "Jim\'s first day at Cook State School" (The full English text).\n\n## NEXT STEPS\nTranslate the entire story artifact into Spanish.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='2f9397c5-f712-48e0-aa78-decab52b97d1'),
              AIMessage(content='El morral se sentía imposiblemente pesado, cargado no por libros, sino por el tamaño

In [15]:
from IPython.display import Markdown
Markdown(final_response["messages"][-1].content)

Based on the conversation summary and the story provided:

*   **What is your name?** I do not know your name, as it was not mentioned in our conversation.
*   **What was the boy's name?** The boy's name was **Jim**.
*   **What was the name of the school?** The name of the school was **Cook State School**.

Oh no! it didn't store the user's name in the summary. Let's take a look at the first item in 'messages' to see what has made it to the summary

In [17]:
Markdown(final_response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to translate the previously written story about a boy named Jim's first day at Cook State School from English into Spanish.

## SUMMARY
The user initially tasked the AI with writing a creative story. The core parameters were: a 500-word story about a boy named Jim's first day at Cook State School. This story was successfully written and provided by the AI. The current and immediate goal is to translate this completed narrative into Spanish.

## ARTIFACTS
One story artifact was created: "Jim's first day at Cook State School" (The full English text).

## NEXT STEPS
Translate the entire story artifact into Spanish.

These are the token usages

In [18]:
total_tokens = 0
for m in final_response["messages"]:
    if hasattr(m, "usage_metadata"):
        print(m.usage_metadata)
        total_tokens += m.usage_metadata["total_tokens"]
print(f"Total tokens used: {total_tokens}")

{'input_tokens': 919, 'output_tokens': 1372, 'total_tokens': 2291}
{'input_tokens': 1852, 'output_tokens': 1418, 'total_tokens': 3270}
{'input_tokens': 2046, 'output_tokens': 381, 'total_tokens': 2427}
Total tokens used: 7988


The first item in messages now has this additional metadata that indicates it is a summary

In [19]:
final_response["messages"][0].additional_kwargs

{'lc_source': 'summarization'}